# Ridership Time-Series Downsampling


## Load and transform data


In [ ]:
from google.colab import drive
import numpy as np
import pandas as pd
import os

drive.mount('/content/drive')

csvsFolderPath = '/content/drive/My Drive/CIS 4500 HW/CIS4500FinalProject/UltraProcessedDataCSVsRECENT'

pd.read_csv(os.path.join(csvsFolderPath, 'ridership2020Dataframe.csv'))

Mounted at /content/drive


,station_complex_id,ridership,transfers,transit_timestamp_millis
0,501,110,19,1577840400000
1,502,5,2,1577840400000
2,1,172,0,1577840400000
3,10,400,0,1577840400000
4,100,103,9,1577840400000
...,...,...,...,...
3390607,97,178,8,1609416000000
3390608,98,185,7,1609416000000
3390609,99,77,0,1609416000000
3390610,TRAM1,74,8,1609416000000


In [ ]:
def cleanDf(df, reduction_factor=4):
    # Convert milliseconds to datetime
    df['datetime'] = pd.to_datetime(df['transit_timestamp_millis'], unit='ms')

    # Sort by station and time to ensure sequential order
    df = df.sort_values(by=['station_complex_id', 'datetime'])

    # Extract month to aggregate within each month
    df['month'] = df['datetime'].dt.month

    # Create a group key for every reduction_factor rows within each station's timeline per month
    df['block'] = df.groupby(['station_complex_id', 'month']).cumcount() // reduction_factor

    # Aggregate the chunks of reduction_factor: sum the counts, keep the first timestamp
    df_reduced = df.groupby(['station_complex_id', 'month', 'block']).agg({
        'ridership': 'sum',
        'transfers': 'sum',
        'transit_timestamp_millis': 'first',
        'datetime': 'first'
    }).reset_index()

    # Drop the temporary columns as they are no longer needed
    df_reduced = df_reduced.drop(columns=['block', 'month', 'datetime'])

    # Assign the sorted dataframe back to the variable
    df_reduced = df_reduced.sort_values(by='transit_timestamp_millis').reset_index(drop=True)

    return df_reduced


## Save reduced datasets


In [ ]:
#save to drive
importCsvsPath = '/content/drive/My Drive/CIS 4500 HW/CIS4500FinalProject/UltraProcessedDataCSVsRECENT'
exportCsvsPath = '/content/drive/My Drive/CIS 4500 HW/CIS4500FinalProject/HyperProcessedDataCSVs'

for year in range(2020, 2025):
  csv_file_path = os.path.join(importCsvsPath, "ridership" + str(year) + "Dataframe.csv")
  df = pd.read_csv(csv_file_path)
  reduced_df = cleanDf(df, 5)

  print(reduced_df.shape)
  print(reduced_df.head())
  print("Saving to drive...")
  reduced_df.to_csv(os.path.join(exportCsvsPath, "ridership" + str(year) + "Dataframe.csv"), index=False)
  print("Saved to drive.")

(680231, 4)
  station_complex_id  ridership  transfers  transit_timestamp_millis
0                 22        177          0             1577836800000
1                378        188         26             1577836800000
2                286        465         53             1577836800000
3                377        220          2             1577836800000
4                 49        115          3             1577836800000
Saving to drive...
Saved to drive.
(712082, 4)
  station_complex_id  ridership  transfers  transit_timestamp_millis
0                198         41          0             1609459200000
1                216        144          0             1609459200000
2                154         64          4             1609459200000
3                617        368         12             1609459200000
4                351         91          0             1609459200000
Saving to drive...
Saved to drive.
(736592, 4)
  station_complex_id  ridership  transfers  transit_timestamp_mill

In [ ]:
# csv_file_path = os.path.join('/content/drive/My Drive/CIS 4500 HW/CIS4500FinalProject/UltraProcessedDataCSVsRECENT', "ridership2021Dataframe.csv")
# og_df = pd.read_csv(csv_file_path)
# print(og_df.shape)
# df = cleanDf(og_df, 10)
# df['datetime'] = pd.to_datetime(df['transit_timestamp_millis'], unit='ms')

# df['month'] = df['datetime'].dt.month

# # Create a pivot table: stations as rows, months as columns, counting occurrences
# pivot_df = df.pivot_table(index='station_complex_id', columns='month', values='transit_timestamp_millis', aggfunc='count', fill_value=0)

# print("Station counts per month:")
# # Filter using the index instead of a standard column
# display(pivot_df)#[pivot_df.index.astype(str) == "301"])

# # Check for any 0 counts (missing data for a station in a specific month)
# zero_counts = pivot_df[pivot_df == 0].stack()
# if not zero_counts.empty:
#     print("\nFound stations with 0 counts in certain months:")
#     display(zero_counts)
# else:
#     print("\nNo stations have 0 counts! Every station is present in every month.")
